# kaggloop — Colab GPU runner (`colab_kaggloop`)

Self-contained GPU worker for the kaggloop compute bridge. It watches a **Google Drive**
folder that your laptop and Colab both see, runs each training job enqueued by
`python -m kloop.colab submit` on the GPU, and writes results back for
`python -m kloop.colab ingest` to pull.

**One-time setup**
1. Runtime → Change runtime type → **GPU** (T4/L4/A100).
2. On the laptop, point kaggloop at the same Drive folder (already wired in
   `.claude/settings.json`):
   `KLOOP_COLAB_QUEUE=.../My Drive/kaggloop/queue`, `KLOOP_COLAB_RESULTS=.../My Drive/kaggloop/results`.
3. Put `kaggle.json` where this notebook can read it — either at
   `My Drive/kaggloop/kaggle.json` or via the upload prompt in the mount cell.

**Run**: Runtime → **Run all**, approve the Drive mount, and leave the last cell polling.
Parallel workers? Give each its own queue folder (`queue-a`, `queue-b`, …) — never point two
runners at one queue (Drive sync is eventual-consistent, so the atomic-rename claim can race).


In [ ]:
# 1. GPU sanity check
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "no nvidia-smi")
try:
    import torch
    print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as e:
    print("torch not imported:", e)


In [ ]:
# 2. Mount Drive + config: point QUEUE/RESULTS at the shared kaggloop folder, set up kaggle.json.
import os, shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# EDIT this if your bridge folder differs (must match KLOOP_COLAB_QUEUE/RESULTS on the laptop).
BRIDGE  = Path("/content/drive/MyDrive/kaggloop")
QUEUE   = BRIDGE / "queue"
RESULTS = BRIDGE / "results"
DATA    = Path("/content/kaggle_data")   # local cache for competition data (not on Drive)
for d in (QUEUE, RESULTS, DATA):
    d.mkdir(parents=True, exist_ok=True)
print("queue   =", QUEUE)
print("results =", RESULTS)

# kaggle.json: prefer one dropped in the Drive bridge; else prompt to upload. Kept in the
# session only (~/.kaggle), never written back to Drive.
kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
src = BRIDGE / "kaggle.json"
if src.exists():
    shutil.copy(src, kdir / "kaggle.json")
    print("kaggle.json <- Drive bridge")
elif not (kdir / "kaggle.json").exists():
    from google.colab import files
    print("Upload kaggle.json:")
    up = files.upload()
    for name, data in up.items():
        (kdir / "kaggle.json").write_bytes(data)
os.chmod(kdir / "kaggle.json", 0o600)
# kaggle CLI is preinstalled on Colab; install if missing.
try:
    import kaggle  # noqa: F401
except Exception:
    subprocess.run(["pip", "install", "-q", "kaggle"], text=True)
print("config ready")


In [ ]:
# 3. kaggloop worker logic (embedded verbatim from colab/worker.py).
#!/usr/bin/env python3
"""kaggloop Colab worker — the GPU half of the compute bridge.

Runs inside a Google Colab notebook (GPU runtime). It watches a shared
directory (a Google Drive folder that both your laptop and Colab can see),
picks up jobs enqueued locally by ``kloop.colab``, runs each training
entrypoint on the GPU, and writes results back for the laptop to ingest.

One job folder is self-contained::

    $QUEUE/<job_id>/job.json   {job_id, run_id, competition, entrypoint, args,
                               requirements, timeout, gpu, ...}
    $QUEUE/<job_id>/code/      snapshot of the campaign's experiments/code

The worker, per job:
  1. claims it (renames job.json -> job.running so it isn't double-run),
  2. ensures the competition data is downloaded+unzipped to a cached data dir
     (via the kaggle API — Colab has the bandwidth and the GPU),
  3. runs ``python <entrypoint> [args]`` with cwd = the job's code dir and
     env KLOOP_DATA_DIR / KLOOP_OUT_DIR set, capturing stdout+stderr,
  4. collects the metric (a ``{"metric": ...}`` stdout line or out/metric.json),
  5. writes $RESULTS/<job_id>/{result.json, run.log, artifacts/*}.

All console output is English (code, comments, and stdout).

Usage (inside Colab, after mounting Drive and uploading kaggle.json)::

    python worker.py --queue /content/drive/MyDrive/kaggloop/queue \
                     --results /content/drive/MyDrive/kaggloop/results \
                     --data /content/kaggle_data --once   # or --loop
"""

from __future__ import annotations

import argparse
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path


def log(msg: str) -> None:
    print(f"[worker {time.strftime('%H:%M:%S')}] {msg}", flush=True)


# --------------------------------------------------------------------------- data

def ensure_competition_data(competition: str, data_root: Path) -> Path:
    """Download + unzip a competition's data once, cached under data_root/<comp>."""
    dest = data_root / competition
    if dest.exists() and any(dest.iterdir()):
        return dest
    dest.mkdir(parents=True, exist_ok=True)
    if not competition:
        log("No competition specified; skipping data download.")
        return dest
    log(f"Downloading competition data: {competition}")
    rc = subprocess.run(
        ["kaggle", "competitions", "download", "-c", competition, "-p", str(dest)],
        text=True,
    ).returncode
    if rc != 0:
        log("x Data download failed (check rules acceptance / kaggle.json).")
        return dest
    for z in dest.glob("*.zip"):
        log(f"Extracting: {z.name}")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        # Free the archive right after extraction so we don't hold zip + unzipped
        # data at once (large competitions can be tens of GB; Colab disk is finite).
        try:
            z.unlink()
            log(f"Removed archive: {z.name}")
        except OSError:
            pass
    return dest


# --------------------------------------------------------------------------- run

def _find_metric(stdout: str, out_dir: Path):
    # Prefer out/metric.json; fall back to the last {"metric": ...} stdout line.
    mj = out_dir / "metric.json"
    if mj.exists():
        try:
            obj = json.loads(mj.read_text())
            if isinstance(obj, dict) and "metric" in obj:
                return obj
        except json.JSONDecodeError:
            pass
    found = None
    for line in stdout.splitlines():
        line = line.strip()
        if line.startswith("{") and line.endswith("}"):
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict) and "metric" in obj:
                found = obj
    return found


def run_job(job_dir: Path, results_root: Path, data_root: Path) -> None:
    job = json.loads((job_dir / "job.json").read_text())
    job_id = job["job_id"]
    log(f"Job start: {job_id}  (entry={job['entrypoint']})")

    # Claim it so a second worker / re-run won't pick it up.
    running = job_dir / "job.running"
    try:
        (job_dir / "job.json").rename(running)
    except OSError:
        log("Job already claimed; skipping.")
        return

    out_dir = results_root / job_id
    art_dir = out_dir / "artifacts"
    art_dir.mkdir(parents=True, exist_ok=True)
    code_dir = job_dir / "code"

    data_dir = ensure_competition_data(job.get("competition", ""), data_root)

    # Optional per-job requirements.
    req = job.get("requirements")
    if req:
        req_path = code_dir / req
        if req_path.exists():
            log(f"pip install -r {req}")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "-r", str(req_path)], text=True)

    env = dict(os.environ)
    env["KLOOP_DATA_DIR"] = str(data_dir)
    env["KLOOP_OUT_DIR"] = str(art_dir)

    cmd = [sys.executable, job["entrypoint"], *job.get("args", [])]
    t0 = time.time()
    timed_out = False
    try:
        proc = subprocess.run(
            cmd, cwd=str(code_dir), env=env, text=True,
            capture_output=True, timeout=int(job.get("timeout", 3600)),
        )
        rc, out, err = proc.returncode, proc.stdout, proc.stderr
    except subprocess.TimeoutExpired as e:
        timed_out = True
        rc = -1
        out = e.stdout or ""
        err = (e.stderr or "") + f"\n[worker] TIMEOUT after {job.get('timeout')}s"
    dur = time.time() - t0

    (out_dir / "run.log").write_text(
        f"$ {' '.join(cmd)}\n# cwd={code_dir}\n# rc={rc} dur={dur:.1f}s\n"
        f"\n----- STDOUT -----\n{out or ''}\n----- STDERR -----\n{err or ''}"
    )

    metric = _find_metric(out or "", art_dir)
    is_buggy = rc != 0 or timed_out or metric is None
    artifacts = sorted(str(p.relative_to(out_dir)) for p in art_dir.rglob("*") if p.is_file())
    result = {
        "job_id": job_id,
        "run_id": job.get("run_id"),
        "ok": not is_buggy,
        "is_buggy": is_buggy,
        "returncode": rc,
        "timed_out": timed_out,
        "duration_s": round(dur, 1),
        "metric": metric,
        "artifacts": artifacts,
        "stderr_tail": (err or "")[-1000:],
        "finished": time.strftime("%Y-%m-%d_%H-%M-%S"),
    }
    (out_dir / "result.json").write_text(json.dumps(result, indent=2, ensure_ascii=False))
    # Mark the queue job done.
    try:
        running.rename(job_dir / "job.done")
    except OSError:
        pass
    status = "ok" if not is_buggy else "failed(buggy)"
    log(f"Job done: {job_id}  {status}  metric={metric}  ({dur:.0f}s)")


def pending_jobs(queue_root: Path):
    if not queue_root.exists():
        return []
    return sorted(d for d in queue_root.iterdir()
                  if d.is_dir() and (d / "job.json").exists())


In [ ]:
# 4. Run the worker loop with a RUNNER_ALIVE heartbeat. Leave this cell running.
import time, traceback
from pathlib import Path

DATA = Path("/content/kaggle_data")
print("runner up; watching", QUEUE, "— leave this cell running")
while True:
    try:
        (QUEUE.parent / "RUNNER_ALIVE").write_text(str(time.time()))  # heartbeat
        for jd in pending_jobs(QUEUE):
            print("[run]", jd.name)
            try:
                run_job(jd, RESULTS, DATA)
            except Exception:
                traceback.print_exc()
    except Exception:
        traceback.print_exc()
    time.sleep(20)
